# 02 — Vérification du dataset MVTec AD
**Projet :** IndustrialDefect AI · **US-02-01 (IDAI-9)** · **ST-9-01 (IDAI-38)**

Objectif : vérifier que les catégories `bottle`, `hazelnut` et `transistor` sont présentes,
correctement structurées (`train/`, `test/`, `ground_truth/`), complètes et lisibles.

Source : [MVTec AD sur Kaggle](https://www.kaggle.com/datasets/ipythonx/mvtec-ad) — licence CC BY-NC-SA 4.0

In [1]:
from pathlib import Path
import sys

# ===== Configuration (chemins configurables) =====
KAGGLE_INPUT = Path("/kaggle/input")
DATASET_ROOT = None          # None = détection automatique ; sinon, mettre un chemin
CATEGORIES = ["bottle", "hazelnut", "transistor"]
REQUIRED_DIRS = ["train", "test", "ground_truth"]


def find_mvtec_root(base: Path, category: str = "bottle") -> Path:
    """Cherche le dossier parent qui contient <category>/train, test et ground_truth."""
    candidates = [
        p.parent
        for p in base.rglob(category)
        if p.is_dir() and all((p / d).is_dir() for d in REQUIRED_DIRS)
    ]
    assert len(candidates) == 1, f"Racines MVTec trouvées : {candidates}"
    return candidates[0]


if DATASET_ROOT is None:
    DATASET_ROOT = find_mvtec_root(KAGGLE_INPUT)
DATASET_ROOT = Path(DATASET_ROOT)

print("Python            :", sys.version.split()[0])
print("Racine du dataset :", DATASET_ROOT)
print("\nContenu de la racine :")
for p in sorted(DATASET_ROOT.iterdir()):
    print("  -", p.name + ("/" if p.is_dir() else ""))

Python            : 3.13.18
Racine du dataset : /kaggle/input/datasets/ipythonx/mvtec-ad

Contenu de la racine :
  - bottle/
  - cable/
  - capsule/
  - carpet/
  - grid/
  - hazelnut/
  - leather/
  - license.txt
  - metal_nut/
  - pill/
  - readme.txt
  - screw/
  - tile/
  - toothbrush/
  - transistor/
  - wood/
  - zipper/


## Étape 3 — Structure des catégories
Règles vérifiées : dossiers `train/test/ground_truth` présents, `train` limité à `good`,
`test` contenant `good`, et types de défauts identiques entre `test` et `ground_truth`.

In [2]:
import pandas as pd


def list_subdirs(path: Path) -> list[str]:
    """Noms des sous-dossiers, triés (liste vide si le dossier n'existe pas)."""
    return sorted(p.name for p in path.iterdir() if p.is_dir()) if path.is_dir() else []


structure_rows = []
defect_types = {}

for cat in CATEGORIES:
    cat_dir = DATASET_ROOT / cat
    train_sub = list_subdirs(cat_dir / "train")
    test_sub = list_subdirs(cat_dir / "test")
    gt_sub = list_subdirs(cat_dir / "ground_truth")
    defects = [t for t in test_sub if t != "good"]
    defect_types[cat] = defects

    checks = {
        "1. dossier présent": cat_dir.is_dir(),
        "2. train/test/ground_truth": all((cat_dir / d).is_dir() for d in REQUIRED_DIRS),
        "3. train = good uniquement": train_sub == ["good"],
        "4. test contient good": "good" in test_sub,
        "5. ground_truth = défauts du test": gt_sub == defects,
    }
    structure_rows.append({"catégorie": cat, **checks, "défauts": ", ".join(defects)})

    print(f"=== {cat} ===")
    print("  train/        :", train_sub)
    print("  test/         :", test_sub)
    print("  ground_truth/ :", gt_sub)
    for name, ok in checks.items():
        print(f"  {'✅' if ok else '❌'} {name}")
    print()

structure_df = pd.DataFrame(structure_rows).set_index("catégorie")
check_cols = [c for c in structure_df.columns if c != "défauts"]
all_ok = bool(structure_df[check_cols].all().all())
print("STRUCTURE VALIDE POUR LES 3 CATÉGORIES :", all_ok)
assert all_ok, "Structure invalide : voir les ❌ ci-dessus"
structure_df

=== bottle ===
  train/        : ['good']
  test/         : ['broken_large', 'broken_small', 'contamination', 'good']
  ground_truth/ : ['broken_large', 'broken_small', 'contamination']
  ✅ 1. dossier présent
  ✅ 2. train/test/ground_truth
  ✅ 3. train = good uniquement
  ✅ 4. test contient good
  ✅ 5. ground_truth = défauts du test

=== hazelnut ===
  train/        : ['good']
  test/         : ['crack', 'cut', 'good', 'hole', 'print']
  ground_truth/ : ['crack', 'cut', 'hole', 'print']
  ✅ 1. dossier présent
  ✅ 2. train/test/ground_truth
  ✅ 3. train = good uniquement
  ✅ 4. test contient good
  ✅ 5. ground_truth = défauts du test

=== transistor ===
  train/        : ['good']
  test/         : ['bent_lead', 'cut_lead', 'damaged_case', 'good', 'misplaced']
  ground_truth/ : ['bent_lead', 'cut_lead', 'damaged_case', 'misplaced']
  ✅ 1. dossier présent
  ✅ 2. train/test/ground_truth
  ✅ 3. train = good uniquement
  ✅ 4. test contient good
  ✅ 5. ground_truth = défauts du test

STRUCTUR

,1. dossier présent,2. train/test/ground_truth,3. train = good uniquement,4. test contient good,5. ground_truth = défauts du test,défauts
catégorie,,,,,,
bottle,True,True,True,True,True,"broken_large, broken_small, contamination"
hazelnut,True,True,True,True,True,"crack, cut, hole, print"
transistor,True,True,True,True,True,"bent_lead, cut_lead, damaged_case, misplaced"


## Étape 4 — Effectifs des images et des masques
Comparaison avec les effectifs officiels de MVTec AD (Bergmann et al., CVPR 2019)
et vérification d'un masque par image défectueuse.

In [3]:
# Effectifs officiels (Bergmann et al., CVPR 2019 — MVTec AD)
EXPECTED = {
    "bottle":     {"train/good": 209, "test/good": 20,
                   "broken_large": 20, "broken_small": 22, "contamination": 21},
    "hazelnut":   {"train/good": 391, "test/good": 40,
                   "crack": 18, "cut": 17, "hole": 18, "print": 17},
    "transistor": {"train/good": 213, "test/good": 60,
                   "bent_lead": 10, "cut_lead": 10, "damaged_case": 10, "misplaced": 10},
}
IMG_EXT = ".png"


def count_png(folder: Path) -> int:
    """Nombre de fichiers .png dans un dossier (0 s'il n'existe pas)."""
    return sum(1 for f in folder.glob(f"*{IMG_EXT}") if f.is_file()) if folder.is_dir() else 0


rows = []
for cat in CATEGORIES:
    cat_dir = DATASET_ROOT / cat
    exp = EXPECTED[cat]
    # Images normales : pas de masque attendu
    for split in ["train", "test"]:
        rows.append({"catégorie": cat, "split": split, "type": "good",
                     "images": count_png(cat_dir / split / "good"),
                     "masques": None, "attendu": exp[f"{split}/good"]})
    # Images défectueuses : un masque par image
    for d in defect_types[cat]:
        rows.append({"catégorie": cat, "split": "test", "type": d,
                     "images": count_png(cat_dir / "test" / d),
                     "masques": count_png(cat_dir / "ground_truth" / d),
                     "attendu": exp.get(d)})

counts_df = pd.DataFrame(rows).astype({"masques": "Int64"})
counts_df["images_ok"] = counts_df["images"] == counts_df["attendu"]
counts_df["masques_ok"] = (
    counts_df["masques"].isna() | (counts_df["masques"] == counts_df["images"])
).fillna(True).astype(bool)

summary_df = (
    counts_df.assign(
        normales=lambda df: df["images"].where(df["type"] == "good", 0),
        anormales=lambda df: df["images"].where(df["type"] != "good", 0),
        masques=lambda df: df["masques"].fillna(0),
    )
    .groupby("catégorie")[["normales", "anormales", "masques"]].sum().astype(int)
)
summary_df.loc["TOTAL"] = summary_df.sum()

print("=== Effectifs détaillés ===")
print(counts_df.to_string(index=False))
print("\n=== Résumé par catégorie ===")
print(summary_df.to_string())

all_counts_ok = bool(counts_df["images_ok"].all() and counts_df["masques_ok"].all())
print("\nEFFECTIFS CONFORMES AUX CHIFFRES OFFICIELS :", all_counts_ok)
assert all_counts_ok, "Écart détecté : voir les lignes où images_ok ou masques_ok = False"

=== Effectifs détaillés ===
 catégorie split          type  images  masques  attendu  images_ok  masques_ok
    bottle train          good     209     <NA>      209       True        True
    bottle  test          good      20     <NA>       20       True        True
    bottle  test  broken_large      20       20       20       True        True
    bottle  test  broken_small      22       22       22       True        True
    bottle  test contamination      21       21       21       True        True
  hazelnut train          good     391     <NA>      391       True        True
  hazelnut  test          good      40     <NA>       40       True        True
  hazelnut  test         crack      18       18       18       True        True
  hazelnut  test           cut      17       17       17       True        True
  hazelnut  test          hole      18       18       18       True        True
  hazelnut  test         print      17       17       17       True        True
transistor t

## Étape 5 — Intégrité des images et des masques
Chaque image est ouverte et décodée ; chaque image défectueuse doit avoir un masque
de même nom, de même taille, binaire (0/255) et non vide.

In [4]:
import numpy as np
from PIL import Image

img_rows, problems = [], []

for cat in CATEGORIES:
    cat_dir = DATASET_ROOT / cat
    folders = [("train", "good"), ("test", "good")] + [("test", d) for d in defect_types[cat]]
    for split, typ in folders:
        for img_path in sorted((cat_dir / split / typ).glob(f"*{IMG_EXT}")):
            row = {"catégorie": cat, "split": split, "type": typ, "fichier": img_path.name}
            # 1) L'image s'ouvre et se décode entièrement
            try:
                with Image.open(img_path) as im:
                    im.load()
                    row.update(largeur=im.width, hauteur=im.height, mode=im.mode)
            except Exception as e:
                problems.append(f"Image illisible : {img_path} ({e})")
                continue
            # 2) Pour un défaut : masque présent, même taille, binaire et non vide
            if typ != "good":
                mask_path = cat_dir / "ground_truth" / typ / f"{img_path.stem}_mask{IMG_EXT}"
                if not mask_path.is_file():
                    problems.append(f"Masque manquant : {mask_path}")
                else:
                    try:
                        with Image.open(mask_path) as m:
                            m.load()
                            arr = np.array(m)
                        if m.size != (row["largeur"], row["hauteur"]):
                            problems.append(f"Taille masque ≠ image : {mask_path}")
                        if not set(np.unique(arr)) <= {0, 255}:
                            problems.append(f"Masque non binaire : {mask_path}")
                        if arr.max() == 0:
                            problems.append(f"Masque vide : {mask_path}")
                    except Exception as e:
                        problems.append(f"Masque illisible : {mask_path} ({e})")
            img_rows.append(row)

images_df = pd.DataFrame(img_rows)
print("Images ouvertes avec succès :", len(images_df))

print("\n=== Formats des images par catégorie ===")
print(images_df.groupby(["catégorie", "largeur", "hauteur", "mode"]).size()
      .rename("nb_images").to_string())

print(f"\nProblèmes détectés : {len(problems)}")
for p in problems[:20]:
    print("  ❌", p)

integrity_ok = len(problems) == 0
print("\nINTÉGRITÉ VALIDÉE :", integrity_ok)
assert integrity_ok, "Problèmes d'intégrité : voir la liste ci-dessus"

Images ouvertes avec succès : 1106

=== Formats des images par catégorie ===
catégorie   largeur  hauteur  mode
bottle      900      900      RGB     292
hazelnut    1024     1024     RGB     501
transistor  1024     1024     RGB     313

Problèmes détectés : 0

INTÉGRITÉ VALIDÉE : True


## Étape 6 — Export des résultats
Les tableaux d'effectifs et un rapport JSON (source, licence, environnement, résultats des
contrôles) sont enregistrés dans `/kaggle/working/mvtec_check/` et conservés par Save Version.

In [5]:
import json, platform
from datetime import datetime, timezone
import PIL

OUTPUT_DIR = Path("/kaggle/working/mvtec_check")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

formats_df = (images_df.groupby(["catégorie", "largeur", "hauteur", "mode"])
              .size().rename("nb_images").reset_index())

counts_df.to_csv(OUTPUT_DIR / "counts_detailed.csv", index=False)
summary_df.to_csv(OUTPUT_DIR / "counts_summary.csv")
formats_df.to_csv(OUTPUT_DIR / "image_formats.csv", index=False)

report = {
    "project": "IndustrialDefect AI",
    "jira": {"us": "IDL-9 (US-02-01)", "subtask": "IDL-10 (ST-02-01-01)"},
    "notebook": "02-mvtec-dataset-check",
    "executed_at_utc": datetime.now(timezone.utc).isoformat(timespec="seconds"),
    "dataset": {
        "name": "MVTec AD",
        "kaggle_url": "https://www.kaggle.com/datasets/ipythonx/mvtec-ad",
        "kaggle_owner": "ipythonx (Innat)",
        "original_source": "https://www.mvtec.com/company/research/datasets/mvtec-ad",
        "license": "CC BY-NC-SA 4.0",
        "citation": "Bergmann et al., CVPR 2019",
        "root_path": str(DATASET_ROOT),
    },
    "categories": CATEGORIES,
    "defect_types": defect_types,
    "summary": summary_df.to_dict(orient="index"),
    "checks": {
        "structure_ok": all_ok,
        "counts_match_official": all_counts_ok,
        "integrity_ok": integrity_ok,
        "problems": problems,
    },
    "environment": {
        "python": sys.version.split()[0],
        "pandas": pd.__version__,
        "numpy": np.__version__,
        "pillow": PIL.__version__,
        "platform": platform.platform(),
    },
}
with open(OUTPUT_DIR / "dataset_check_report.json", "w", encoding="utf-8") as f:
    json.dump(report, f, indent=2, ensure_ascii=False, default=int)

print("=== Fichiers de résultats ===")
for f in sorted(OUTPUT_DIR.iterdir()):
    print(f"  {f.name:28s} {f.stat().st_size:>7} octets")

print("\n=== BILAN ST-02-01-01 ===")
print("  Structure valide        :", all_ok)
print("  Effectifs officiels     :", all_counts_ok)
print("  Intégrité des fichiers  :", integrity_ok)
assert all_ok and all_counts_ok and integrity_ok
print("\n✅ Les 3 catégories MVTec AD sont complètes, bien structurées et intactes.")

=== Fichiers de résultats ===
  counts_detailed.csv              759 octets
  counts_summary.csv               113 octets
  dataset_check_report.json       1639 octets
  image_formats.csv                121 octets

=== BILAN ST-02-01-01 ===
  Structure valide        : True
  Effectifs officiels     : True
  Intégrité des fichiers  : True

✅ Les 3 catégories MVTec AD sont complètes, bien structurées et intactes.
